# Online Fraud Detection — Model Training

This notebook trains and compares Logistic Regression, Decision Tree, and Random Forest models for the fraud-detection capstone. It includes basic EDA, stratified splitting, preprocessing, SMOTE, cross-validation, MLflow tracking, evaluation plots, and saving the selected pipeline.

**Before running:** place your CSV at `data/transactions.csv` relative to the project root. In VS Code, open the project folder, select the Python environment where `requirements.txt` is installed, and run cells from top to bottom.

> The full dataset has over 6.3 million rows. SMOTE and cross-validation can be memory- and time-intensive. For a first smoke test, set `SAMPLE_FRAC = 0.05` in the configuration cell; use `1.0` for the full dataset. Record clearly in your report if you use a sample.

## 1. Imports and configuration

In [ ]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, f1_score, precision_score, recall_score,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Assumes the notebook is running from the project root.
ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "transactions.csv"
TARGET_REQUESTED = "isFraud"
SAMPLE_FRAC = 1.0  # Change to 0.05 for a quick smoke test; 1.0 uses all rows.
RANDOM_STATE = 42

REPORTS_DIR = ROOT / "reports"
MODELS_DIR = ROOT / "models"
REPORTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

## 2. Load and inspect the dataset

In [ ]:
def find_target(dataframe, requested):
    candidates = [requested, "isFraud", "Is_Fraud", "is_fraud"]
    for column in candidates:
        if column in dataframe.columns:
            return column
    raise ValueError(
        f"Could not find target column. Requested={requested!r}; "
        f"available columns={list(dataframe.columns)}"
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Put the CSV there or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
target = find_target(df, TARGET_REQUESTED)
df = df.drop_duplicates().copy()
df = df.dropna(subset=[target])

if SAMPLE_FRAC < 1.0:
    df, _ = train_test_split(
        df, train_size=SAMPLE_FRAC, random_state=RANDOM_STATE, stratify=df[target]
    )

print("Dataset shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head(20))
print("\nSummary statistics:\n", df.describe(include="all").T.head(30))

## 3. Exploratory data analysis: target imbalance

In [ ]:
target_counts = df[target].value_counts(dropna=False).sort_index()
target_proportions = df[target].value_counts(normalize=True, dropna=False).sort_index()
print("Target counts:\n", target_counts)
print("\nTarget proportions:\n", target_proportions)

ax = target_counts.plot(kind="bar", title="Class distribution")
ax.set_xlabel("Target class (0 = legitimate, 1 = fraud)")
ax.set_ylabel("Transaction count")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "class_distribution.png", dpi=160)
plt.show()

## 4. Prepare features and split the data

In [ ]:
# Exclude the target, high-cardinality identifiers, and the rule-based flag.
# Keeping isFlaggedFraud could leak information from an existing fraud rule.
leakage_or_id = [
    target, "nameOrig", "nameDest", "transaction_id", "TransactionID",
    "isFlaggedFraud", "IsFlaggedFraud"
]
X = df.drop(columns=[column for column in leakage_or_id if column in df.columns]).copy()
y = pd.to_numeric(df[target], errors="raise").astype(int)

unique_y = set(y.unique())
if not unique_y.issubset({0, 1}) or y.nunique() != 2:
    raise ValueError(f"Target must contain both binary classes 0 and 1; found {unique_y}")

# If a timestamp field exists, derive useful time features.
for column in list(X.columns):
    if column.lower() in {"timestamp", "datetime", "transaction_time"}:
        parsed = pd.to_datetime(X[column], errors="coerce")
        X[f"{column}_hour"] = parsed.dt.hour
        X[f"{column}_dayofweek"] = parsed.dt.dayofweek
        X = X.drop(columns=[column])

X = X.dropna(axis=1, how="all")
if X.shape[1] == 0:
    raise ValueError("No usable features remain after removing target/IDs.")

categorical = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numerical = [column for column in X.columns if column not in categorical]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Features:", list(X.columns))
print("Categorical columns:", categorical)
print("Numerical columns:", numerical)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Training class counts before resampling:\n", y_train.value_counts())

## 5. Define preprocessing and models

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numerical),
    ("cat", categorical_pipe, categorical),
])

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight=None),
    "decision_tree": DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE,
        class_weight="balanced"
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=150, min_samples_leaf=2, n_jobs=-1,
        random_state=RANDOM_STATE, class_weight="balanced_subsample"
    ),
}

min_class_count = int(y_train.value_counts().min())
if min_class_count < 2:
    raise ValueError("Too few minority-class examples for validation; use a larger sample.")
n_splits = min(3, min_class_count)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print(f"Using {n_splits}-fold stratified cross-validation.")

## 6. Configure MLflow tracking

In [ ]:
# Local SQLite tracking database. Later, this can be replaced with the Azure ML
# workspace's MLflow tracking URI when the Azure connection is configured.
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("online-fraud-detection")
print("MLflow tracking URI:", tracking_uri)

## 7. Train, cross-validate, and evaluate models

In [ ]:
results = []
trained_pipelines = {}

for name, estimator in models.items():
    print(f"\n===== Training {name} =====")
    # SMOTE is inside the imbalanced-learn pipeline so it is applied only to
    # training folds, not to the held-out test set.
    pipeline = ImbPipeline([
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, min_class_count - 1))),
        ("model", estimator),
    ])

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("dataset_rows", len(df))
        mlflow.log_param("feature_count_before_encoding", X.shape[1])
        mlflow.log_param("sample_fraction", SAMPLE_FRAC)
        mlflow.log_param("smote", True)

        cv_scores = cross_validate(
            pipeline, X_train, y_train, cv=cv,
            scoring={
                "f1": "f1",
                "precision": "precision",
                "recall": "recall",
                "average_precision": "average_precision",
            },
            n_jobs=1, error_score="raise"
        )
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": float(accuracy_score(y_test, predictions)),
            "precision": float(precision_score(y_test, predictions, zero_division=0)),
            "recall": float(recall_score(y_test, predictions, zero_division=0)),
            "f1": float(f1_score(y_test, predictions, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_test, probabilities)),
            "pr_auc": float(average_precision_score(y_test, probabilities)),
            "cv_f1_mean": float(np.mean(cv_scores["test_f1"])),
            "cv_precision_mean": float(np.mean(cv_scores["test_precision"])),
            "cv_recall_mean": float(np.mean(cv_scores["test_recall"])),
            "cv_pr_auc_mean": float(np.mean(cv_scores["test_average_precision"])),
        }
        mlflow.log_metrics(metrics)
        mlflow.log_dict(
            {"classification_report": classification_report(
                y_test, predictions, output_dict=True, zero_division=0)},
            "classification_report.json"
        )

        plot_specs = [
            ("confusion_matrix", lambda ax: ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)),
            ("roc_curve", lambda ax: RocCurveDisplay.from_predictions(y_test, probabilities, ax=ax)),
            ("pr_curve", lambda ax: PrecisionRecallDisplay.from_predictions(y_test, probabilities, ax=ax)),
        ]
        for plot_name, draw in plot_specs:
            fig, ax = plt.subplots()
            draw(ax)
            fig.tight_layout()
            plot_path = REPORTS_DIR / f"{name}_{plot_name}.png"
            fig.savefig(plot_path, dpi=160)
            plt.show()
            plt.close(fig)
            mlflow.log_artifact(str(plot_path))

        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        print(json.dumps(metrics, indent=2))
        results.append({"model": name, **metrics})
        trained_pipelines[name] = pipeline

## 8. Compare models and save the selected pipeline

In [ ]:
results_df = pd.DataFrame(results).sort_values("cv_pr_auc_mean", ascending=False)
results_path = REPORTS_DIR / "model_comparison.csv"
results_df.to_csv(results_path, index=False)

# Select using cross-validation PR-AUC, not the held-out test PR-AUC.
best_name = str(results_df.iloc[0]["model"])
best_pipeline = trained_pipelines[best_name]

model_path = MODELS_DIR / "fraud_pipeline.joblib"
joblib.dump({
    "pipeline": best_pipeline,
    "feature_columns": list(X.columns),
    "target_column": target,
    "selected_model": best_name,
}, model_path)

with open(REPORTS_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump({"selected_model": best_name, "models": results}, file, indent=2)

print("Selected model based on mean CV PR-AUC:", best_name)
print("Saved model:", model_path)
print("Saved comparison:", results_path)
display(results_df)

## 9. Final evaluation and saved-model check

In [ ]:
final_predictions = best_pipeline.predict(X_test)
final_probabilities = best_pipeline.predict_proba(X_test)[:, 1]
print("Selected model:", best_name)
print("Test PR-AUC:", round(average_precision_score(y_test, final_probabilities), 5))
print("\nClassification report:\n")
print(classification_report(y_test, final_predictions, zero_division=0))

saved = joblib.load(model_path)
in_memory = best_pipeline.predict(X_test.iloc[:10])
reloaded = saved["pipeline"].predict(X_test.iloc[:10])
if not np.array_equal(in_memory, reloaded):
    raise RuntimeError("Saved/reloaded model prediction parity check failed.")
print("Saved model reload check passed.")

## 10. Output checklist

After all cells complete, check that these files exist in the project folder:

- `models/fraud_pipeline.joblib`
- `reports/model_comparison.csv`
- `reports/metrics.json`
- `reports/class_distribution.png`
- `reports/*_confusion_matrix.png`, `reports/*_roc_curve.png`, and `reports/*_pr_curve.png`
- `mlflow.db` (local MLflow tracking database)

**Important capstone note:** this notebook compares three baseline models. Add hyperparameter tuning (for example, `GridSearchCV` or `RandomizedSearchCV`) before considering the modeling requirement complete. The full-data SMOTE workflow can be resource-intensive; if it exhausts memory, we should revise the resampling strategy rather than repeatedly rerun it unchanged.